In [3]:
# import env file, make sure it contains your kaggle token

from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())   # finds ../.env from notebooks/
print("token loaded:", "KAGGLE_API_TOKEN" in os.environ)

token loaded: True


In [4]:
# download dataset

import kagglehub
from pathlib import Path

DATA_FOLDER = "./../data/raw/"

# Download latest version
if not Path(DATA_FOLDER).exists():
    path = kagglehub.competition_download('favorita-grocery-sales-forecasting', 
                                      output_dir=DATA_FOLDER)

    print("Path to competition files:", path)

In [5]:
# extract files

from pathlib import Path
import py7zr

EXTRACTED_FOLDER = Path(DATA_FOLDER) / "extracted"

if not Path(EXTRACTED_FOLDER).exists():
    folder = Path(DATA_FOLDER)
    for archive in sorted(folder.glob("*.csv.7z")):
        out_dir = archive.with_suffix("")
        print(f"Extracting {archive.name} -> {out_dir.name}")
        try:
            with py7zr.SevenZipFile(archive, mode="r") as z:
                z.extractall(path=EXTRACTED_FOLDER)
        except Exception as e:
            print(f"  Failed: {e}")


In [38]:
# convert train.csv to parquet

import polars as pl

PROCESSED_FOLDER = Path("./../data/processed/")

if not Path(PROCESSED_FOLDER / "train.parquet").exists():
    (
    pl.scan_csv(
        EXTRACTED_FOLDER / "train.csv",
        schema_overrides={
            "store_nbr": pl.Int8,# 1 to 54
            "item_nbr": pl.Int32, # up to ~2.1 million
            "unit_sales": pl.Float32, # decimals
            "onpromotion": pl.Boolean, # True / False / empty
        },
        try_parse_dates=True
    )
    .drop("id")
    .sink_parquet(PROCESSED_FOLDER / "train.parquet", mkdir=True)
    )

train = pl.read_parquet(PROCESSED_FOLDER / "train.parquet")
train.head(800)

date,store_nbr,item_nbr,unit_sales,onpromotion
date,i8,i32,f32,bool
2013-01-01,25,103665,7.0,null
2013-01-01,25,105574,1.0,null
2013-01-01,25,105575,2.0,null
2013-01-01,25,108079,1.0,null
2013-01-01,25,108701,1.0,null
…,…,…,…,…
2013-01-02,1,314881,7.0,null
2013-01-02,1,315176,29.0,null
2013-01-02,1,315178,7.0,null


In [43]:
# read other datafdrames

import pandas as pd

items = pd.read_csv(EXTRACTED_FOLDER / "items.csv")
print(items.head())

transactions = pd.read_csv(EXTRACTED_FOLDER / "transactions.csv")
print(transactions.head())

stores = pd.read_csv(EXTRACTED_FOLDER / "stores.csv")
print(stores.head())

holidays = pd.read_csv(EXTRACTED_FOLDER / "holidays_events.csv")
print(holidays.head())

oil = pd.read_csv(EXTRACTED_FOLDER / "oil.csv")
print(oil.head())

   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1
         date  store_nbr  transactions
0  2013-01-01         25           770
1  2013-01-02          1          2111
2  2013-01-02          2          2358
3  2013-01-02          3          3487
4  2013-01-02          4          1922
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4
         date     type    locale locale_name        

# Things to do

* df_train -> wochentag und kalendarmonat als feature erzeugen

* df_train -> mit items verbinden

* stores -> verbinden mit holidays

* df_train -> verbinden mit oil price

* items -> family als class

## Common

* check for outliers

* check for non values

In [44]:
# check test data

test = pd.read_csv(EXTRACTED_FOLDER / "test.csv")
print(test.head())

          id        date  store_nbr  item_nbr  onpromotion
0  125497040  2017-08-16          1     96995        False
1  125497041  2017-08-16          1     99197        False
2  125497042  2017-08-16          1    103501        False
3  125497043  2017-08-16          1    103520        False
4  125497044  2017-08-16          1    103665        False
